# Sentiment Analysis Training
This notebook demonstrates how to load the dataset, train the model, and evaluate it.

## 1. Setup & Imports

In [2]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
import os
import logging

os.environ["MLFLOW_TRACKING_URI"] = "http://localhost:5001"
os.environ["MLFLOW_EXPERIMENT_NAME"] = "Sentiment_Analysis"
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

model_name = "vinai/phobert-base"
output_dir = "../models/sentiment"

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

RuntimeError: Failed to import transformers.trainer because of the following error (look up to see its traceback):
Failed to import transformers.integrations.integration_utils because of the following error (look up to see its traceback):
Failed to import transformers.modeling_tf_utils because of the following error (look up to see its traceback):
cannot import name 'runtime_version' from 'google.protobuf' (/Users/quanghuy/miniconda3/lib/python3.12/site-packages/google/protobuf/__init__.py)

## 2. Load and Prepare Dataset

In [ ]:
logger.info("Loading `sepidmnorozy/Vietnamese_sentiment` Dataset from HuggingFace...")
dataset = load_dataset("sepidmnorozy/Vietnamese_sentiment")
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples['text'], padding="max_length", max_length=128, truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)
tokenized_datasets = tokenized_datasets.remove_columns(["text"])

train_dataset = tokenized_datasets['train'].shuffle(seed=42).select(range(min(500, len(tokenized_datasets['train']))))
eval_dataset = tokenized_datasets['validation'].shuffle(seed=42).select(range(min(50, len(tokenized_datasets['validation']))))

print(f"Train size: {len(train_dataset)}, Eval size: {len(eval_dataset)}")

## 3. Train Model

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

def evaluate_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = torch.argmax(torch.tensor(logits), axis=-1).numpy()
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='macro')
    return {"accuracy": acc, "f1": f1}

training_args = TrainingArguments(
    output_dir=output_dir,
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    push_to_hub=False,
    report_to="mlflow"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=evaluate_metrics,
)

trainer.train()
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
logger.info("Training complete and model saved.")

## 4. Evaluation & Testing

In [ ]:
# Test the trained model
from transformers import pipeline

sentiment_pipeline = pipeline("text-classification", model=output_dir, tokenizer=output_dir)

samples = [
    "Sản phẩm này rất tốt, tôi rất thích!",
    "Chất lượng quá kém, không đáng tiền.",
    "Bình thường, không có gì đặc sắc."
]

for text in samples:
    result = sentiment_pipeline(text)
    print(f"Text: {text}\nPredicted: {result}\n")